### Задача 1 

Для примера применнения цели возьмём достаточно простую задачу из электротехники. Представим что мы хотим сделать универсальный решатель задач по электротехнике для цепей постоянного тока. В качестве кокнретного примера представим, что у нас есть электрическая цепь постоянного тока с напряжением питания 24 Вольта и двумя последовательно подключенными резисторами 100 и 1500 Ом. 

Наша цепь может работать в 2 режимах в зависимости от потребляемой мощности - "нормальный" и "перегрузка". Цепь может работать только в одном из этих режимов.

Мы хотим выяснить одно из допустимых значений тока, при которых цепь будет работать в нормально режиме. Сначала запишем переменные, которыми будем оперировать при решении задачи:

In [1]:
from z3 import *

U = Real("U") # Напряжение питания
R1 = Real("R1") # Сопротивление первого резистора
R2 = Real("R2") # Сопротивление второго резистора
R = Real("R") # Общее сопротивление цепи
I = Real("I") # Общий ток в цепи
P = Real("P") # Общая потребляемая мощность

normal, overload = Bools('normal overload')

Goal представляет собой набор логических ограничений, которым должно удовлетворять решение задачи. Все ограничения рассматриваются как конъюнкция (логическое И).

Сформулируем ограниченитя для такой цепи:

In [2]:
g = Goal()

g.add(U == 24)
g.add(normal == True)
g.add(R1 == 100, R2 == 1500)

g.add(R == R1 + R2)
g.add(I == U / R)
g.add(P == U * I)

g.add(Xor(normal, overload)) # Может быть только один из режимов
g.add(Implies(overload, P >= 50)) # Перегрузка при мощности больше 50 Ватт
g.add(Implies(normal, And(P < 50, P > 0))) # Явно указываем критерии нормального режимиа

g

[U == 24,
 normal == True,
 R1 == 100,
 R2 == 1500,
 R == R1 + R2,
 I == U/R,
 P == U*I,
 Xor(normal, overload),
 Implies(overload, P >= 50),
 Implies(normal, And(P < 50, P > 0))]

И так, видно набор критериев на основе которых солвер должен рассчитать наши искомые значения. Мы можем сразу передать эти критерии солверу для поиска решения, а можем воспользоваться тактикой propagate-values, чтобы упростить наши ограничения после подстановки известных исходных данных (тактика не решает задачу, а лишь распространяет уже известные значения по всем ограничениям):

In [3]:
t = Tactic('propagate-values')
t(g)

[[U == 24,
  normal,
  R1 == 100,
  R2 == 1500,
  R == 1600,
  I == 3/200,
  P == 9/25,
  Not(overload)]]

Вместо U, R1 и R2 были везде подставлены константы. Теперь солверу даже не нужно решать уравнения, все значеия получены до вычислений солвером после подстановки. Обратимся к солверу за моделью:

In [4]:
s = Solver()
s.add(t(g).as_expr())

s.check()


sat

In [5]:
s.model()

[R2 = 1500,
 I = 3/200,
 R = 1600,
 overload = False,
 U = 24,
 P = 9/25,
 R1 = 100,
 normal = True]

Теперь мы знаем, что при таких исходных параметрах цепь работает в нормальном режиме. Также известно значение тока.

### Задача 2

В SMT есть множество тактик, некоторые помогают упростить ограничения или избавиться от лишних переменных, другие декомпозировать задачу на более мелки подцели которые можно решить по отдельности. 

Предположим что есть программа которая компилируется в 3 режимах Debug, Sandbox и Prod. Одновременно может быть только один режим. Для Debug режима лог выводится в локальный файл, для Sandbox лог выводится в Elastic dev, а для Prod в Elastic prod. Запишем цель такой задачи и её требования:

In [6]:
debug, sand, prod, log_file, log_e_dev, log_e_prod = Bools("debug sand prod log_file log_e_dev log_e_prod")

goal = Goal()

goal.add(Or(
    And(debug, log_file), 
    And(sand, log_e_dev), 
    And(prod, log_e_prod)))

goal

[Or(And(debug, log_file),
    And(sand, log_e_dev),
    And(prod, log_e_prod))]

Определим тактику для декомпозиции задачи на несколько подзадач:

In [7]:
tsc = Tactic("split-clause")
tres = tsc(goal)

def print_subgoals(res):
    for i, subgoal in enumerate(res, start=1):
        print(f"=== Subgoal {i} ===")
        print(subgoal)
        print()

print_subgoals(tres)

=== Subgoal 1 ===
[debug, log_file]

=== Subgoal 2 ===
[sand, log_e_dev]

=== Subgoal 3 ===
[prod, log_e_prod]



Видим, что вместо одной формулы с тремя альтернативными сценариями получили три отдельные подцели. Каждая такая подцель соответствует одному варианту конфигурации (Debug, Sandbox или Prod), поэтому дальнейший анализ можно выполнять для каждой ветви независимо. Также стоит отметить что эта тактика не разбираетсяв смысле формулы, а только разбивает дизъюнкцию Or на подцели которые можно решить отдельно.

### Задача 3

Солвер позволяет комбинировать тактики при помощи комбинаторов. Для примера рассмотрим усложненную версию задачи 2:

Представим, что помимо логов есть еще ограничения на авторизацию, сбор пользовательских метрик, и размер нативных логов. Под каждое доп. ограничение определим новые переменные, ограничения выразим через импликации:

In [8]:
debug, sand, prod = Bools("debug sand prod")
log_file, log_e_dev, log_e_prod = Bools("log_file log_e_dev log_e_prod")
is_auth_available, collect_metrics = Bools("is_auth_available collect_metrics")
native_logs_lines = Int('native_logs_lines')

goal = Goal()

goal.add(Or(
    And(debug, log_file), 
    And(sand, log_e_dev), 
    And(prod, log_e_prod)))

goal.add(Implies(debug, Not(is_auth_available)))
goal.add(Implies(debug, Not(collect_metrics)))
goal.add(Implies(debug, native_logs_lines <= 5000))

goal.add(Implies(sand, Not(is_auth_available)))
goal.add(Implies(sand, collect_metrics))
goal.add(Implies(sand, native_logs_lines <= 1000))

goal.add(Implies(prod, is_auth_available))
goal.add(Implies(prod, collect_metrics))
goal.add(Implies(prod, native_logs_lines <= 200))

goal

[Or(And(debug, log_file),
    And(sand, log_e_dev),
    And(prod, log_e_prod)),
 Implies(debug, Not(is_auth_available)),
 Implies(debug, Not(collect_metrics)),
 Implies(debug, native_logs_lines <= 5000),
 Implies(sand, Not(is_auth_available)),
 Implies(sand, collect_metrics),
 Implies(sand, native_logs_lines <= 1000),
 Implies(prod, is_auth_available),
 Implies(prod, collect_metrics),
 Implies(prod, native_logs_lines <= 200)]

In [9]:
comb = Then('split-clause', 'propagate-values')
#comb = Tactic('split-clause')
print_subgoals(comb(goal))

=== Subgoal 1 ===
[debug,
 Not(is_auth_available),
 Not(collect_metrics),
 native_logs_lines <= 5000,
 Not(sand),
 Not(prod),
 log_file]

=== Subgoal 2 ===
[sand,
 Not(debug),
 Not(is_auth_available),
 collect_metrics,
 native_logs_lines <= 1000,
 Not(prod),
 log_e_dev]

=== Subgoal 3 ===
[prod,
 Not(debug),
 Not(sand),
 is_auth_available,
 collect_metrics,
 native_logs_lines <= 200,
 log_e_prod]



После применения комбинатора Then результат работы split-clause передается тактике propagate-values. Поскольку в каждой подцели уже известно какой режим активен, то propagate-values вычисляет значения импликаций и заменяя их соответствующими следствиями. В результате вместо условных ограничений в подцелях осаются непосредственные факты, например Not(is_auth_available) или native_logs_lines <= 5000.

### Задача 4

Для примера возьмём задачу настройки регистра 8 битного микроконтроллера (Ардуино например :) ). Пусть нам надо убедиться, что UART должен работать в режиме 2, внешние прерывания должны быть запрещены, энергосбережение должно быть отключено. Зафикисируем цель:

In [10]:
register = BitVec("register", 8)

goal = Goal()

goal.add(
    register & 0b00000011 == 0b00000010,  # UART = 2
    register & 0b00001000 == 0,           # Interrupt OFF
    register & 0b00010000 == 0,           # Sleep OFF
)

goal

[register & 3 == 2, register & 8 == 0, register & 16 == 0]

Подготовим свой солвер на основе комбинатора:

In [11]:
bit_solver = Then(
    'bit-blast',
    'sat'
).solver()

bit_solver.add(goal.as_expr())

print(bit_solver.check())
print(bit_solver.model())

sat
[register = 2]


После применения bit-blast операции над битовыми векторами преобразуются в эквивалентную булеву формулу, в которой каждый бит рассматривается как отдельная логическая переменная. Полученная формула передается тактике sat, которая подбирает комбинацию значений битов.

### Задача 5

Усовершенствуем наш ардуино-солвер. Предположим что нам надо не только искать и устанавливать значения регистров, но и проверять допустимые диапазоны датчиков, а также калибровать их. Это разные задачи которые требуют разных стратегий решения. Для этого будем использовать анализ целей через probe. 

Необходимо проверить, что контроллер подлкючено согласно следующим условиям:
- Датчик DHT22 должен быть подключен;
- Показания должны выводиться на OLED-дисплей;
- SD-карта в данной конфигурации не используется;
- Данные передаются по Wi-Fi;
- Для хранения времени используется RTC DS3231;
- Звуковой сигнализатор отключен;
- RGB-индикатор отключен;
- Релейный модуль подключен;

In [35]:
port_config = BitVec("port_config", 8)

goal_controller = Goal()

goal_controller.add(
    port_config & 0b00000001 == 0b00000001, # DHT22 (датчик температуры и влажности) включен
    port_config & 0b00000010 == 0b00000010, # OLED включен
    port_config & 0b00000100 == 0, # SD-карта отключена
    port_config & 0b00001000 == 0b00001000, # Wi-Fi включен
    port_config & 0b00010000 == 0b00010000, # RTC (часы) подключен
    port_config & 0b00100000 == 0, # Buzzer отключен
    port_config & 0b01000000 == 0, # RGB отключен
    port_config & 0b10000000 == 0b10000000 # Реле подключено
)

Проверить, относится ли цель к логике битовых векторов, можно при помощи пробы is-qfbv:

In [36]:
is_qfbv_probe = Probe('is-qfbv')
p(goal)

1.0

Значение 1.0 говорит что цель относится к логике битовых векторов, подготовим для этого стратегию решения:

In [37]:
bit_strategy = Then( "bit-blast","sat")

Теперь подготовим цель для рповерки датчика:

In [38]:
temperature = Int("temperature")
humidity = Int("humidity")

goal_linear_sensor = Goal()

goal_linear_sensor.add(
    temperature >= -20,
    temperature <= 60,
    humidity >= 0,
    humidity <= 100,
)

И соответсвующую универсальную стратегию для таких целей:

In [39]:
lia_strategy = Then("simplify", "solve-eqs", "smt")

Эта тактика получилась универсальной, поэтому задачи калибровки можно также выполнять ей. Однако для примера подготовим третий тип солвера для нелинейных уравнений:

In [40]:
nonlia_strategy = Then("simplify", "smt")

Для проверки линейности/нелинейности критериев цели используем is-qflia:

In [41]:
is_qflia_probe = Probe("is-qflia")

print(is_qflia_probe(goal_linear_sensor))

1.0


Построим итоговый солвер:

In [44]:
lia_or_nonlia = If(is_qflia_probe, lia_strategy, nonlia_strategy)
solver = If(is_qfbv_probe, bit_strategy, lia_or_nonlia).solver()

solver.add(goal_controller.as_expr())

solver.check()

sat

In [45]:
solver.model()

[port_config = 155]

In [46]:
solver.reset()
solver.add(goal_linear_sensor.as_expr())
solver.check()

sat

In [47]:
solver.model()

[humidity = 0, temperature = 0]

Солвер выбрал подходящую стратегию и решил.